# FinAnomX – Report Notebook





In [15]:
# 1.1 Import thư viện và cấu hình đường dẫn notebook
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# --------------------------------------------------
# XÁC ĐỊNH ĐÚNG THƯ MỤC GỐC CỦA PROJECT
# --------------------------------------------------
CURRENT_DIR = Path.cwd().resolve()

if (CURRENT_DIR / "data").exists() and (CURRENT_DIR / "src").exists():
    # Trường hợp notebook đang chạy từ thư mục gốc FinAnomX
    ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists() and (CURRENT_DIR.parent / "src").exists():
    # Trường hợp notebook đang chạy từ thư mục FinAnomX/notebooks
    ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        f"Không xác định được thư mục gốc project.\n"
        f"CURRENT_DIR = {CURRENT_DIR}\n"
        f"Hãy kiểm tra notebook đang được chạy từ đâu."
    )

# --------------------------------------------------
# ĐƯỜNG DẪN DỮ LIỆU
# --------------------------------------------------
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_RESULTS = ROOT / "data" / "results"
RUNS_DIR = ROOT / "runs" / "latest"

# --------------------------------------------------
# THƯ MỤC LƯU HÌNH/BẢNG CỦA NOTEBOOK
# --------------------------------------------------
REPORT_DIR = ROOT / "notebooks" / "report_outputs"
FIG_DIR = REPORT_DIR / "figures"
TABLE_DIR = REPORT_DIR / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("CURRENT_DIR:", CURRENT_DIR)
print("ROOT:", ROOT)
print("DATA_RAW exists:", DATA_RAW.exists())
print("DATA_PROCESSED exists:", DATA_PROCESSED.exists())
print("DATA_RESULTS exists:", DATA_RESULTS.exists())
print("REPORT_DIR:", REPORT_DIR)

CURRENT_DIR: C:\Users\Abc\Downloads\finanomx\notebooks
ROOT: C:\Users\Abc\Downloads\finanomx
DATA_RAW exists: True
DATA_PROCESSED exists: True
DATA_RESULTS exists: True
REPORT_DIR: C:\Users\Abc\Downloads\finanomx\notebooks\report_outputs


In [16]:
# 1.2 Helper functions + LaTeX-like style + load all report data
from pathlib import Path
import json
import textwrap
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import transforms

warnings.filterwarnings("ignore")

# --------------------------------------------------
# REPORT WINDOW: bám sát web overview mặc định
# --------------------------------------------------
REPORT_START = pd.Timestamp("2023-01-01")
REPORT_END = pd.Timestamp("2024-12-31")

# --------------------------------------------------
# CAPTION DƯỚI HÌNH
# - True  : vẽ caption ngay dưới figure
# - False : chừa khoảng trắng phía dưới để bạn tự chèn caption trong LaTeX
# --------------------------------------------------
SHOW_CAPTION_IN_FIG = False

# --------------------------------------------------
# STYLE: LaTeX-like + pastel nhẹ
# --------------------------------------------------
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",
    "axes.edgecolor": "#D7DEE8",
    "axes.linewidth": 0.8,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.frameon": False,
    "legend.fontsize": 10,
    "figure.dpi": 120,
})

PALETTE = {
    # Màu theo ticker
    "amzn": "#8FB9E8",
    "msft": "#95D1C3",
    "nvda": "#F4C095",
    "tsla": "#C7B6E6",

    # Màu subtype
    "price-spike-up": "#F6C7A4",
    "price-spike-down": "#F1B5BE",
    "volume-burst": "#C9BDE2",
    "mixed": "#A9D9D0",
    "other": "#A9DEF9",

    # Màu nhóm feature
    "price": "#A9C4E2",
    "volume": "#E9C7AB",
    "volatility": "#E7C5CF",
    "technical": "#9FD1C8",
    "macro": "#CFC5E2",

    # Màu phụ
    "threshold": "#A7D8CF",
    "event": "#E6B8B7",
    "neutral": "#7D8CA3",
    "grid": "#E9EDF2",
    "text": "#3F4B5B",
}

TICKER_COLORS = {
    "AMZN": PALETTE["amzn"],
    "MSFT": PALETTE["msft"],
    "NVDA": PALETTE["nvda"],
    "TSLA": PALETTE["tsla"],
}

SUBTYPE_ORDER = ["price-spike-up", "price-spike-down", "volume-burst", "mixed", "other"]
GROUP_ORDER = ["price", "volume", "volatility", "technical", "macro"]
TICKER_ORDER = ["AMZN", "MSFT", "NVDA", "TSLA"]

# --------------------------------------------------
# HELPER: đọc parquet an toàn
# --------------------------------------------------
def read_parquet_safe(path):
    try:
        return pd.read_parquet(path)
    except Exception:
        import pyarrow.parquet as pq
        return pq.read_table(path).to_pandas()

# --------------------------------------------------
# HELPER: lưu figure PNG only
# - caption đặt dưới figure
# - reserve_bottom để chừa khoảng trắng phía dưới
# --------------------------------------------------
def save_figure(fig, filename, caption=None, reserve_bottom=0.14):
    if caption is not None and SHOW_CAPTION_IN_FIG:
        fig.text(
            0.5, 0.02, caption,
            ha="center", va="bottom",
            fontsize=13, color=PALETTE["text"]
        )

    out_path = FIG_DIR / filename
    fig.savefig(out_path, dpi=300)
    plt.close(fig)
    print(f"[Saved figure] {out_path}")
    return out_path

# --------------------------------------------------
# HELPER: lưu table CSV only
# --------------------------------------------------
def save_table_csv(df, filename):
    out_path = TABLE_DIR / filename
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"[Saved table] {out_path}")
    return out_path

# --------------------------------------------------
# HELPER: style trục nhẹ, thoáng
# --------------------------------------------------
def soften_axes(ax, grid_axis="y"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#D7DEE8")
    ax.spines["bottom"].set_color("#D7DEE8")
    ax.tick_params(colors="#546172")
    ax.grid(False)

    if grid_axis in ("x", "both"):
        ax.grid(axis="x", color=PALETTE["grid"], linewidth=0.8)
    if grid_axis in ("y", "both"):
        ax.grid(axis="y", color=PALETTE["grid"], linewidth=0.8)

    return ax

# --------------------------------------------------
# LOAD DATA
# --------------------------------------------------
raw = read_parquet_safe(DATA_RAW / "raw_merged.parquet")
features = read_parquet_safe(DATA_PROCESSED / "features.parquet")
news_raw = read_parquet_safe(DATA_RESULTS / "news_enriched_raw.parquet")
news = read_parquet_safe(DATA_RESULTS / "news_enriched.parquet")

anomalies = pd.read_csv(DATA_RESULTS / "anomalies.csv", parse_dates=["date"], low_memory=False)
event_attr = pd.read_csv(DATA_RESULTS / "event_attribution.csv", parse_dates=["peak_date", "seendate"], low_memory=False)
event_cf = pd.read_csv(DATA_RESULTS / "event_counterfactual.csv", parse_dates=["peak_date"], low_memory=False)
eval_explain = pd.read_csv(DATA_RESULTS / "eval_explain.csv")
model_comp = pd.read_csv(DATA_RESULTS / "model_comparison_adapt.csv")
xai_feature = pd.read_csv(DATA_RESULTS / "xai_feature.csv", parse_dates=["date"])

chat_evidence_path = DATA_RESULTS / "chat_evidence.jsonl"
chat_evidence_count = sum(1 for _ in open(chat_evidence_path, "r", encoding="utf-8"))

raw["date"] = pd.to_datetime(raw["date"])
features["date"] = pd.to_datetime(features["date"])
anomalies["date"] = pd.to_datetime(anomalies["date"])
xai_feature["date"] = pd.to_datetime(xai_feature["date"])

peak_all = anomalies.query("is_episode_peak == 1").copy()

peak_analysis = peak_all[
    (peak_all["date"] >= REPORT_START) &
    (peak_all["date"] <= REPORT_END)
].copy()

grounded_all = peak_all[peak_all["top_event_title"].notna()].copy()
grounded_analysis = peak_analysis[peak_analysis["top_event_title"].notna()].copy()

raw_analysis = raw[
    (raw["date"] >= REPORT_START) &
    (raw["date"] <= REPORT_END)
].copy()

anomalies_analysis = anomalies[
    (anomalies["date"] >= REPORT_START) &
    (anomalies["date"] <= REPORT_END)
].copy()

print("raw:", raw.shape)
print("features:", features.shape)
print("anomalies:", anomalies.shape)
print("peak_analysis:", peak_analysis.shape)
print("grounded_analysis:", grounded_analysis.shape)
print("chat_evidence_count:", chat_evidence_count)

raw: (6956, 12)
features: (6716, 28)
anomalies: (6716, 190)
peak_analysis: (25, 190)
grounded_analysis: (25, 190)
chat_evidence_count: 6716


In [17]:
# 2.1 Table 1 - Related works and FinAnomX contribution
related_work_df = pd.DataFrame([
    {
        "Study": "Li and Fan [1]",
        "Main focus": "Detect heterogeneous anomalies in financial networks and attribute driving mechanisms.",
        "Event / news grounding": "No article-level event grounding",
        "Counterfactual validation": "No",
        "Key limitation vs FinAnomX": "Does not retrieve concrete article-level evidence for a stock-day anomaly.",
        "What FinAnomX adds": "Grounded event retrieval for peak anomalies plus rebuttal-style counterfactual checks.",
    },
    {
        "Study": "Hu et al. (MindTS) [2]",
        "Main focus": "Multimodal time-series anomaly detection via time-text semantic alignment.",
        "Event / news grounding": "Not finance-specific article-level grounding",
        "Counterfactual validation": "No",
        "Key limitation vs FinAnomX": "Not centered on identifying a concrete market event as the likely cause of one stock anomaly.",
        "What FinAnomX adds": "Finance-specific grounding, top-1 event ranking, and robustness checks for the proposed cause.",
    },
    {
        "Study": "Çalık et al. [3]",
        "Main focus": "Stock-price prediction and interpretability using DLinear / LTSNet / Transformer + SHAP / LIME.",
        "Event / news grounding": "No",
        "Counterfactual validation": "No",
        "Key limitation vs FinAnomX": "Provides interpretability for prediction, but not event-grounded anomaly explanation.",
        "What FinAnomX adds": "Moves from model interpretability to evidence-grounded anomaly explanation with rebuttal analysis.",
    },
    {
        "Study": "FinAnomX (this work)",
        "Main focus": "Detect stock anomalies, retrieve plausible event evidence, rank likely causes, and validate them via counterfactual checks.",
        "Event / news grounding": "Yes",
        "Counterfactual validation": "Yes",
        "Key limitation vs FinAnomX": "—",
        "What FinAnomX adds": "Offline-first pipeline, packaged artifacts, dashboard, and grounded chatbot explanations.",
    },
])

save_table_csv(related_work_df, "table_01_related_works_and_contribution.csv")
related_work_df

[Saved table] C:\Users\Abc\Downloads\finanomx\notebooks\report_outputs\tables\table_01_related_works_and_contribution.csv


,Study,Main focus,Event / news grounding,Counterfactual validation,Key limitation vs FinAnomX,What FinAnomX adds
0,Li and Fan [1],Detect heterogeneous anomalies in financial ne...,No article-level event grounding,No,Does not retrieve concrete article-level evide...,Grounded event retrieval for peak anomalies pl...
1,Hu et al. (MindTS) [2],Multimodal time-series anomaly detection via t...,Not finance-specific article-level grounding,No,Not centered on identifying a concrete market ...,"Finance-specific grounding, top-1 event rankin..."
2,Çalık et al. [3],Stock-price prediction and interpretability us...,No,No,"Provides interpretability for prediction, but ...",Moves from model interpretability to evidence-...
3,FinAnomX (this work),"Detect stock anomalies, retrieve plausible eve...",Yes,Yes,—,"Offline-first pipeline, packaged artifacts, da..."


In [18]:
# 2.2 Table 2 - Dataset and artifact summary
dataset_summary = pd.DataFrame([
    {
        "Component": "Raw market + macro panel",
        "Scope / period": f"{raw['ticker'].nunique()} tickers | {raw['date'].min().date()} to {raw['date'].max().date()}",
        "Records": f"{len(raw):,}",
        "Key fields / outputs": "OHLCV + VIX + DGS10 + FEDFUNDS + CPI",
        "Role in report": "Input market-macro backbone used for anomaly screening.",
    },
    {
        "Component": "Processed feature panel",
        "Scope / period": f"{features['ticker'].nunique()} tickers | {features['date'].min().date()} to {features['date'].max().date()}",
        "Records": f"{len(features):,}",
        "Key fields / outputs": "returns, RSI, MACD, rolling volatility, volume z-score, macro diffs",
        "Role in report": "Engineered feature space fed into detectors and XAI.",
    },
    {
        "Component": "News retrieval candidates (raw)",
        "Scope / period": "Peak-centered article search windows",
        "Records": f"{len(news_raw):,}",
        "Key fields / outputs": "article_title, article_summary, article_text, domain, tone",
        "Role in report": "Initial evidence pool before quality filtering.",
    },
    {
        "Component": "Usable enriched news evidence",
        "Scope / period": "After content enrichment and filtering",
        "Records": f"{len(news):,}",
        "Key fields / outputs": "full_text / summary evidence, sentiment, embeddings",
        "Role in report": "Article evidence actually eligible for grounding.",
    },
    {
        "Component": "Episode peaks in analysis window",
        "Scope / period": f"{REPORT_START.date()} to {REPORT_END.date()}",
        "Records": f"{len(peak_analysis):,}",
        "Key fields / outputs": "subtype, main_score, abnormal summary",
        "Role in report": "Peak anomalies used in the main analysis period.",
    },
    {
        "Component": "Grounded peak cases in analysis window",
        "Scope / period": f"{REPORT_START.date()} to {REPORT_END.date()}",
        "Records": f"{len(grounded_analysis):,}",
        "Key fields / outputs": "top_event_title, confidence_score, evidence_text",
        "Role in report": "Cases used for event attribution analysis.",
    },
    {
        "Component": "Counterfactual evaluated peaks",
        "Scope / period": "Grounded peaks with query-ablation checks",
        "Records": f"{len(event_cf):,}",
        "Key fields / outputs": "cf_strength, support_level, top1_changed",
        "Role in report": "Used to test whether the proposed cause is robust.",
    },
    {
        "Component": "Chat-ready artifact",
        "Scope / period": "Packaged explanation records",
        "Records": f"{chat_evidence_count:,}",
        "Key fields / outputs": "numeric summary + event summary + full summary",
        "Role in report": "Supports dashboard and grounded chatbot outputs.",
    },
])

save_table_csv(dataset_summary, "table_02_dataset_artifact_summary.csv")
dataset_summary

[Saved table] C:\Users\Abc\Downloads\finanomx\notebooks\report_outputs\tables\table_02_dataset_artifact_summary.csv


,Component,Scope / period,Records,Key fields / outputs,Role in report
0,Raw market + macro panel,4 tickers | 2018-02-01 to 2024-12-30,"6,956",OHLCV + VIX + DGS10 + FEDFUNDS + CPI,Input market-macro backbone used for anomaly s...
1,Processed feature panel,4 tickers | 2018-04-30 to 2024-12-30,"6,716","returns, RSI, MACD, rolling volatility, volume...",Engineered feature space fed into detectors an...
2,News retrieval candidates (raw),Peak-centered article search windows,236,"article_title, article_summary, article_text, ...",Initial evidence pool before quality filtering.
3,Usable enriched news evidence,After content enrichment and filtering,162,"full_text / summary evidence, sentiment, embed...",Article evidence actually eligible for grounding.
4,Episode peaks in analysis window,2023-01-01 to 2024-12-31,25,"subtype, main_score, abnormal summary",Peak anomalies used in the main analysis period.
5,Grounded peak cases in analysis window,2023-01-01 to 2024-12-31,25,"top_event_title, confidence_score, evidence_text",Cases used for event attribution analysis.
6,Counterfactual evaluated peaks,Grounded peaks with query-ablation checks,25,"cf_strength, support_level, top1_changed",Used to test whether the proposed cause is rob...
7,Chat-ready artifact,Packaged explanation records,"6,716",numeric summary + event summary + full summary,Supports dashboard and grounded chatbot outputs.


In [19]:
# 3.1 Figure 2 - Data coverage and train/validation/test split timeline
split_ranges = features.groupby("split")["date"].agg(["min", "max"]).reset_index()
coverage = raw.groupby("ticker")["date"].agg(["min", "max"]).reset_index().sort_values("ticker")

split_color_map = {
    "train": "#EEF5FC",
    "val": "#F8F1DD",
    "test": "#F8E8EA",
}

fig, ax = plt.subplots(figsize=(14, 4.6))

# Vùng màu split
for _, row in split_ranges.iterrows():
    ax.axvspan(
        row["min"], row["max"],
        color=split_color_map[row["split"]],
        zorder=0
    )

# Text TRAIN / VAL / TEST nằm trên vùng màu, không dùng legend
trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)
for _, row in split_ranges.iterrows():
    mid = row["min"] + (row["max"] - row["min"]) / 2
    ax.text(
        mid, 1.01, row["split"].upper(),
        transform=trans,
        ha="center", va="bottom",
        fontsize=10, color=PALETTE["text"]
    )

# Mỗi ticker 1 màu line riêng
ypos = np.arange(len(coverage))[::-1]
for y, (_, row) in zip(ypos, coverage.iterrows()):
    color = TICKER_COLORS[row["ticker"]]
    ax.hlines(y, row["min"], row["max"], color=color, linewidth=5)
    ax.scatter([row["min"], row["max"]], [y, y], color=color, s=65, zorder=3)

    ax.text(
        row["min"], y + 0.13, row["ticker"],
        fontsize=12, fontweight="semibold",
        color=PALETTE["text"],
        ha="left", va="bottom"
    )

ax.set_yticks([])
ax.set_ylabel("")
ax.set_xlabel("Date")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

soften_axes(ax, grid_axis="x")

fig.subplots_adjust(left=0.06, right=0.99, top=0.86, bottom=0.18)

save_figure(
    fig,
    "fig_02_data_coverage_split_timeline.png",
    caption="Figure 2. Data coverage and train/validation/test split timeline",
    reserve_bottom=0.18,
)

[Saved figure] C:\Users\Abc\Downloads\finanomx\notebooks\report_outputs\figures\fig_02_data_coverage_split_timeline.png


WindowsPath('C:/Users/Abc/Downloads/finanomx/notebooks/report_outputs/figures/fig_02_data_coverage_split_timeline.png')

In [34]:
# 3.2 Figure 3 - Price paths with episode-peak markers in the analysis window
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
axes = axes.flatten()

legend_handles = {}

for ax, ticker in zip(axes, TICKER_ORDER):
    ts = raw_analysis[raw_analysis["ticker"] == ticker].sort_values("date").copy()
    pk = peak_analysis[peak_analysis["ticker"] == ticker].copy()
    pk["subtype_display"] = pk["subtype_display"].fillna("other")

    line_color = TICKER_COLORS[ticker]
    ax.plot(ts["date"], ts["close"], color=line_color, linewidth=1.8)

    for subtype in SUBTYPE_ORDER:
        sub = pk[pk["subtype_display"] == subtype]
        if sub.empty:
            continue

        sc = ax.scatter(
            sub["date"],
            sub["close"],
            s=55,
            color=PALETTE[subtype],
            edgecolor="#5E6B7A",
            linewidth=0.7,
            zorder=3,
        )
        legend_handles[subtype] = sc

    ax.text(
        0.0, 1.03, ticker,
        transform=ax.transAxes,
        ha="left", va="bottom",
        fontsize=12, fontweight="semibold",
        color=PALETTE["text"]
    )

    ax.set_ylabel("Close")
    soften_axes(ax, "y")

    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

for ax in axes:
    for lab in ax.get_xticklabels():
        lab.set_rotation(25)
        lab.set_ha("right")

# Legend ra ngoài plot
ordered_present = [s for s in SUBTYPE_ORDER if s in legend_handles]
fig.legend(
    [legend_handles[s] for s in ordered_present],
    ordered_present,
    ncol=min(5, len(ordered_present)),
    loc="upper center",
    bbox_to_anchor=(0.5, 0.96),
    columnspacing=1.2,
    handletextpad=0.5,
    fontsize=16,
)

fig.subplots_adjust(left=0.07, right=0.99, top=0.82, bottom=0.17, hspace=0.25, wspace=0.12)

save_figure(
    fig,
    "fig_03_anomaly_overview_across_tickers.png",
    caption="Figure 3. Price paths with episode-peak markers in the analysis window",
    reserve_bottom=0.17,
)

[Saved figure] C:\Users\Abc\Downloads\finanomx\notebooks\report_outputs\figures\fig_03_anomaly_overview_across_tickers.png


WindowsPath('C:/Users/Abc/Downloads/finanomx/notebooks/report_outputs/figures/fig_03_anomaly_overview_across_tickers.png')

In [32]:
# 3.3 Figure 4 - Distribution of episode-peak subtypes by ticker (bright pastel + larger labels)

subtype_counts = (
    peak_analysis.assign(subtype_display=peak_analysis["subtype_display"].fillna("other"))
    .groupby(["ticker", "subtype_display"])
    .size()
    .unstack(fill_value=0)
)

subtype_counts = subtype_counts.reindex(index=TICKER_ORDER).fillna(0)
ordered_cols = [c for c in SUBTYPE_ORDER if c in subtype_counts.columns]
subtype_counts = subtype_counts[ordered_cols]

# --------------------------------------------------
# MÀU SÁNG, KHÔNG DÙNG XÁM
# --------------------------------------------------
subtype_colors = {
    "price-spike-up": "#FFD6A5",   # cam pastel
    "price-spike-down": "#FFCAD4", # hồng pastel
    "volume-burst": "#CDB4DB",     # tím pastel
    "mixed": "#B9FBC0",            # xanh lá pastel
    "other": "#A9DEF9",            # xanh dương pastel
}

fig, ax = plt.subplots(figsize=(11.6, 5.8))

x = np.arange(len(subtype_counts.index))
bottom = np.zeros(len(subtype_counts.index))

handles = []
labels = []

for col in subtype_counts.columns:
    vals = subtype_counts[col].values
    bars = ax.bar(
        x,
        vals,
        bottom=bottom,
        color=subtype_colors.get(col, "#A9DEF9"),
        edgecolor="white",
        linewidth=1.0,
        width=0.60,
    )
    bottom += vals
    handles.append(bars[0])
    labels.append(col)

# --------------------------------------------------
# TRỤC VÀ NHÃN
# --------------------------------------------------
ax.set_xticks(x)
ax.set_xticklabels(subtype_counts.index, fontsize=14)
ax.set_ylabel("Number of episode peaks", fontsize=15)

ax.tick_params(axis="y", labelsize=13)
ax.tick_params(axis="x", pad=8)

soften_axes(ax, "y")

# Chừa thêm khoảng trống phía trên để legend không dính vào cột
ymax = max(bottom) if len(bottom) > 0 else 1
ax.set_ylim(0, ymax + 1.6)

# --------------------------------------------------
# LEGEND TO HƠN, DỄ ĐỌC, KHÔNG ĐÈ BIỂU ĐỒ
# --------------------------------------------------
fig.legend(
    handles,
    labels,
    ncol=4,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.99),
    fontsize=16,
    frameon=False,
    handlelength=1.5,
    handletextpad=0.6,
    columnspacing=1.8,
)

fig.subplots_adjust(left=0.09, right=0.99, top=0.74, bottom=0.18)

save_figure(
    fig,
    "fig_04_subtype_distribution.png",
    caption="Figure 4. Distribution of episode-peak subtypes by ticker",
    reserve_bottom=0.18,
)

[Saved figure] C:\Users\Abc\Downloads\finanomx\notebooks\report_outputs\figures\fig_04_subtype_distribution.png


WindowsPath('C:/Users/Abc/Downloads/finanomx/notebooks/report_outputs/figures/fig_04_subtype_distribution.png')

In [22]:
# 4.1 Table 3 - Detection benchmark snapshot
display_name_map = {
    "patchtst_tranad_adapt": "PatchTST + TranAD (adaptive)",
    "chronos_adapt": "Chronos-2 benchmark (adaptive)",
    "tranad_adapt": "TranAD (adaptive)",
    "patchtst_adapt": "PatchTST (adaptive)",
}

role_map = {
    "patchtst_tranad_adapt": "Operational main detector in the packaged pipeline",
    "chronos_adapt": "Benchmark detector used for comparison",
    "tranad_adapt": "Single-model deep detector baseline",
    "patchtst_adapt": "Single-model forecasting detector baseline",
}

benchmark_tbl = model_comp.copy()
benchmark_tbl["Model"] = benchmark_tbl["model"].map(display_name_map).fillna(benchmark_tbl["model"])
benchmark_tbl["Role"] = benchmark_tbl["model"].map(role_map).fillna("—")

benchmark_tbl = benchmark_tbl[
    [
        "Model",
        "Role",
        "auroc",
        "auprc",
        "precision",
        "recall",
        "f1",
        "precision_at_5",
        "precision_at_10",
        "precision_at_20",
    ]
].copy()

for col in ["auroc", "auprc", "precision", "recall", "f1", "precision_at_5", "precision_at_10", "precision_at_20"]:
    benchmark_tbl[col] = benchmark_tbl[col].map(lambda x: f"{x:.4f}")

save_table_csv(benchmark_tbl, "table_03_detection_benchmark_snapshot.csv")
benchmark_tbl

[Saved table] C:\Users\Abc\Downloads\finanomx\notebooks\report_outputs\tables\table_03_detection_benchmark_snapshot.csv


,Model,Role,auroc,auprc,precision,recall,f1,precision_at_5,precision_at_10,precision_at_20
0,Chronos-2 benchmark (adaptive),Benchmark detector used for comparison,0.7887,0.3899,0.2921,0.4643,0.3586,1.0000,0.9000,0.8000
1,TranAD (adaptive),Single-model deep detector baseline,0.8178,0.0696,0.1206,0.4286,0.1882,0.0000,0.0000,0.0000
2,PatchTST + TranAD (adaptive),Operational main detector in the packaged pipe...,0.7980,0.0624,0.1275,0.2321,0.1646,0.0000,0.0000,0.0000
3,PatchTST (adaptive),Single-model forecasting detector baseline,0.5544,0.0308,0.0351,0.2321,0.0610,0.0000,0.0000,0.0000


In [23]:
# 4.2 Table 4 - Explanation / grounding / counterfactual summary
metric_name_map = {
    "faithfulness_pass_rate": "Faithfulness pass rate",
    "faithfulness_mean_drop_abs": "Mean absolute score drop after feature deletion",
    "stability_mean_overlap_top5": "Mean overlap@5 under date jitter",
    "stability_mean_spearman_feat": "Mean Spearman correlation of feature ranks",
    "grounding_mean_confidence_all_ranked": "Mean confidence of all ranked event candidates",
    "grounding_top1_mean_confidence": "Mean confidence of top-1 grounded event",
    "grounding_top1_full_or_summary_share": "Top-1 share using full-text or summary evidence",
    "grounding_top1_full_text_share": "Top-1 share using full-text evidence",
    "grounding_top1_cluster_consensus": "Top-1 event cluster consensus",
    "counterfactual_mean_strength": "Mean counterfactual strength",
    "counterfactual_top1_changed_share": "Share of peaks whose top-1 changes under ablation",
    "counterfactual_strong_support_share": "Share of peaks with strong counterfactual support",
}

metric_comment_map = {
    "faithfulness_pass_rate": "Checks whether removing important features actually lowers the anomaly score.",
    "faithfulness_mean_drop_abs": "Higher drop supports that the explanation is not decorative.",
    "stability_mean_overlap_top5": "Measures whether top features stay similar under small date perturbations.",
    "stability_mean_spearman_feat": "Measures rank consistency of explanations.",
    "grounding_mean_confidence_all_ranked": "Average quality of ranked news candidates before top-1 selection.",
    "grounding_top1_mean_confidence": "Quality of the final proposed cause.",
    "grounding_top1_full_or_summary_share": "Ensures top-1 evidence is not title-only.",
    "grounding_top1_full_text_share": "Shows how often the final cause is backed by full article text.",
    "grounding_top1_cluster_consensus": "Higher means the proposed event is supported by a coherent article cluster.",
    "counterfactual_mean_strength": "Higher means the proposed cause remains robust under rebuttal checks.",
    "counterfactual_top1_changed_share": "Lower is better; too much change would indicate fragile grounding.",
    "counterfactual_strong_support_share": "Direct summary of grounded causes that remain convincing after ablation.",
}

explain_tbl = eval_explain.copy()
explain_tbl["Metric"] = explain_tbl["metric"].map(metric_name_map).fillna(explain_tbl["metric"])
explain_tbl["Value"] = explain_tbl["value"].map(lambda x: f"{x:.4f}")
explain_tbl["Interpretation"] = explain_tbl["metric"].map(metric_comment_map).fillna("—")
explain_tbl = explain_tbl[["Metric", "Value", "Interpretation"]]

extra_rows = pd.DataFrame([
    {
        "Metric": "Grounded peak cases in analysis window",
        "Value": f"{len(grounded_analysis):,}",
        "Interpretation": "Peak anomalies in the main analysis window that received a ranked top-1 event explanation.",
    },
    {
        "Metric": "Counterfactual evaluated peaks",
        "Value": f"{len(event_cf):,}",
        "Interpretation": "Grounded peaks that also received rebuttal-style robustness checks.",
    },
])

explain_tbl = pd.concat([explain_tbl, extra_rows], ignore_index=True)

save_table_csv(explain_tbl, "table_04_explanation_grounding_counterfactual_summary.csv")
explain_tbl

[Saved table] C:\Users\Abc\Downloads\finanomx\notebooks\report_outputs\tables\table_04_explanation_grounding_counterfactual_summary.csv


,Metric,Value,Interpretation
0,Faithfulness pass rate,1.0000,Checks whether removing important features act...
1,Mean absolute score drop after feature deletion,0.2218,Higher drop supports that the explanation is n...
2,Mean overlap@5 under date jitter,0.6286,Measures whether top features stay similar und...
3,Mean Spearman correlation of feature ranks,0.5865,Measures rank consistency of explanations.
4,Mean confidence of all ranked event candidates,0.6891,Average quality of ranked news candidates befo...
5,Mean confidence of top-1 grounded event,0.7948,Quality of the final proposed cause.
6,Top-1 share using full-text or summary evidence,1.0000,Ensures top-1 evidence is not title-only.
7,Top-1 share using full-text evidence,0.8400,Shows how often the final cause is backed by f...
8,Top-1 event cluster consensus,0.9093,Higher means the proposed event is supported b...
9,Mean counterfactual strength,0.8032,Higher means the proposed cause remains robust...


In [31]:
# 4.3 Figure 5 - Feature attribution composition for grounded peaks (larger legend + labels)

xai_top5 = xai_feature[xai_feature["rank"] <= 5].copy()

grounded_keys = grounded_analysis[["ticker", "date"]].drop_duplicates().copy()
xai_grounded = xai_top5.merge(grounded_keys, on=["ticker", "date"], how="inner")

ticker_group_share = (
    xai_grounded.groupby(["ticker", "feature_group"])["importance_norm"]
    .sum()
    .reset_index()
)

ticker_group_share["importance_norm"] = (
    ticker_group_share.groupby("ticker")["importance_norm"]
    .transform(lambda s: s / s.sum())
)

ticker_pivot = (
    ticker_group_share
    .pivot(index="ticker", columns="feature_group", values="importance_norm")
    .fillna(0)
)

present_groups = [g for g in GROUP_ORDER if g in ticker_pivot.columns]
ticker_pivot = ticker_pivot.reindex(index=TICKER_ORDER).fillna(0)[present_groups]

overall_group_share = (
    xai_grounded.groupby("feature_group")["importance_norm"]
    .sum()
    .reindex(present_groups)
    .fillna(0)
)

fig, axes = plt.subplots(
    1, 2, figsize=(13.8, 5.6),
    gridspec_kw={"width_ratios": [1.15, 0.95]}
)

# Left: stacked by ticker
ax = axes[0]
x = np.arange(len(ticker_pivot.index))
bottom = np.zeros(len(ticker_pivot.index))

handles = []
labels = []

for group in present_groups:
    vals = ticker_pivot[group].values
    bars = ax.bar(
        x,
        vals,
        bottom=bottom,
        color=PALETTE[group],
        edgecolor="white",
        linewidth=1.0,
        width=0.62,
    )
    bottom += vals
    handles.append(bars[0])
    labels.append(group)

ax.set_xticks(x)
ax.set_xticklabels(ticker_pivot.index, fontsize=13)
ax.set_ylabel("Share of top-5 normalized importance", fontsize=15)
ax.tick_params(axis="y", labelsize=13)
ax.tick_params(axis="x", pad=6)

ax.text(
    0.5, 1.015, "By ticker",
    transform=ax.transAxes,
    ha="center", va="bottom",
    fontsize=15, fontweight="semibold",
    color=PALETTE["text"]
)

soften_axes(ax, "y")

# Right: overall composition
ax2 = axes[1]
ax2.barh(
    overall_group_share.index,
    overall_group_share.values,
    color=[PALETTE[g] for g in overall_group_share.index],
    edgecolor="white",
    linewidth=1.0,
    height=0.62,
)

ax2.set_xlabel("Cumulative normalized importance", fontsize=15)
ax2.tick_params(axis="x", labelsize=13)
ax2.tick_params(axis="y", labelsize=13)

ax2.text(
    0.5, 1.015, "Overall composition",
    transform=ax2.transAxes,
    ha="center", va="bottom",
    fontsize=15, fontweight="semibold",
    color=PALETTE["text"]
)

soften_axes(ax2, "x")

# --------------------------------------------------
# LEGEND TO HƠN, THOÁNG HƠN, KHÔNG ĐÈ CHART
# --------------------------------------------------
fig.legend(
    handles,
    labels,
    ncol=5,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.99),
    fontsize=16,
    frameon=False,
    handlelength=1.5,
    handletextpad=0.6,
    columnspacing=1.8,
)

fig.subplots_adjust(left=0.07, right=0.99, top=0.78, bottom=0.18, wspace=0.16)

save_figure(
    fig,
    "fig_05_feature_attribution_composition.png",
    caption="Figure 5. Feature attribution composition for grounded peaks",
    reserve_bottom=0.18,
)

[Saved figure] C:\Users\Abc\Downloads\finanomx\notebooks\report_outputs\figures\fig_05_feature_attribution_composition.png


WindowsPath('C:/Users/Abc/Downloads/finanomx/notebooks/report_outputs/figures/fig_05_feature_attribution_composition.png')

In [25]:
# 4.6 Table 5 - Representative grounded case studies
case_df = grounded_analysis.copy()

support_order = {"strong": 0, "moderate": 1}
conf_order = {"high": 0, "medium": 1, "not_evaluated": 2}

case_df["support_rank"] = case_df["cf_support_level"].map(support_order).fillna(9)
case_df["confidence_rank"] = case_df["news_confidence_level"].map(conf_order).fillna(9)
case_df["case_rank_score"] = (
    0.6 * case_df["top_event_confidence_score"].fillna(0) +
    0.4 * case_df["cf_strength"].fillna(0)
)

case_df = case_df.sort_values(
    ["support_rank", "confidence_rank", "case_rank_score"],
    ascending=[True, True, False]
)

case_tbl = (
    case_df.groupby("ticker", group_keys=False)
    .head(2)
    .sort_values(["ticker", "date"])
    .copy()
)

case_tbl = case_tbl[
    [
        "ticker",
        "date",
        "subtype_display",
        "what_was_abnormal_short",
        "xai_top1_feature_pretty",
        "top_event_title",
        "top_event_content_source_type",
        "top_event_confidence_score",
        "cf_support_level",
        "cf_strength",
    ]
].copy()

case_tbl.columns = [
    "Ticker",
    "Peak date",
    "Subtype",
    "Abnormality summary",
    "Top numeric driver",
    "Top grounded event",
    "Source type",
    "Confidence",
    "CF support",
    "CF strength",
]

case_tbl["Peak date"] = pd.to_datetime(case_tbl["Peak date"]).dt.date
case_tbl["Confidence"] = case_tbl["Confidence"].map(lambda x: f"{x:.3f}")
case_tbl["CF strength"] = case_tbl["CF strength"].map(lambda x: f"{x:.3f}")

save_table_csv(case_tbl, "table_05_representative_grounded_case_studies.csv")
case_tbl

[Saved table] C:\Users\Abc\Downloads\finanomx\notebooks\report_outputs\tables\table_05_representative_grounded_case_studies.csv


,Ticker,Peak date,Subtype,Abnormality summary,Top numeric driver,Top grounded event,Source type,Confidence,CF support,CF strength
1384,AMZN,2023-10-27,volume-burst,Volume burst: volume z-score 3.22 with 1-day r...,daily return,Amazon ( AMZN ) Q3 2023 earnings results beat ...,full_text,0.886,strong,0.819
1575,AMZN,2024-08-02,mixed,"Unusual episode lasting 3 day(s), 1-day return...",daily return,Amazon . com ( AMZN ) Rose on Growing Optimism,full_text,0.618,moderate,0.743
3186,MSFT,2024-04-25,volume-burst,Volume burst: volume z-score 3.40 with 1-day r...,abnormal volume,Microsoft ( MSFT ) Continues to Grow Impressively,full_text,0.841,strong,0.765
3255,MSFT,2024-08-05,other,"Unusual episode lasting 3 day(s), 1-day return...",VIX change,Stock Market Falls As Bears Bash Techs ; S & P...,full_text,0.927,strong,0.895
4861,NVDA,2024-04-19,mixed,"Unusual episode lasting 4 day(s), 1-day return...",MACD spread,"Nvidia ETFs Take Center Stage As Bears , Bulls...",full_text,0.862,strong,0.898
4984,NVDA,2024-10-15,other,"Unusual episode lasting 2 day(s), 1-day return...",MACD spread,Nvidia Stock : Blackwell AI Processors On Sche...,full_text,0.889,strong,0.894
6235,TSLA,2023-02-01,other,"Unusual episode lasting 1 day(s), 1-day return...",CPI change,Tesla Stock Forecast : TSLA blasts through $15...,full_text,0.908,strong,0.921
6679,TSLA,2024-11-06,other,"Unusual episode lasting 1 day(s), 1-day return...",daily return,Should You Buy Tesla Stock After Trump Electio...,full_text,0.903,strong,0.863


In [26]:
# 4.7 Figure 8 - Detailed grounded anomaly case study (chart-only)
case_pool = grounded_analysis.copy()
case_pool["case_rank_score"] = (
    0.6 * case_pool["top_event_confidence_score"].fillna(0) +
    0.4 * case_pool["cf_strength"].fillna(0)
)

case_row = case_pool.sort_values("case_rank_score", ascending=False).iloc[0]

ticker = case_row["ticker"]
peak_date = pd.to_datetime(case_row["date"])

window = anomalies[
    (anomalies["ticker"] == ticker) &
    (anomalies["date"].between(peak_date - pd.Timedelta(days=35), peak_date + pd.Timedelta(days=35)))
].copy().sort_values("date")

top_feat = (
    xai_feature[
        (xai_feature["ticker"] == ticker) &
        (xai_feature["date"] == peak_date) &
        (xai_feature["rank"] <= 5)
    ]
    .sort_values("rank")
    .copy()
)

fig = plt.figure(figsize=(12.8, 6.0))
gs = fig.add_gridspec(2, 2, width_ratios=[1.15, 1.0], height_ratios=[1, 1])

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
ax3 = fig.add_subplot(gs[:, 1])

line_color = TICKER_COLORS[ticker]

# Panel 1: price
ax1.plot(window["date"], window["close"], color=line_color, linewidth=1.9)
ax1.scatter(
    [peak_date], [case_row["close"]],
    color=PALETTE["event"], s=70,
    edgecolor="#6B7280", linewidth=0.8, zorder=3
)
ax1.axvline(peak_date, color=PALETTE["event"], linestyle="--", linewidth=1.1)

ax1.text(
    0.0, 1.03, f"{ticker} price around the selected peak",
    transform=ax1.transAxes,
    ha="left", va="bottom",
    fontsize=12, fontweight="semibold",
    color=PALETTE["text"]
)

ax1.set_ylabel("Close")
soften_axes(ax1, "y")

# Panel 2: anomaly score
ax2.plot(
    window["date"], window["main_score"],
    color=PALETTE["neutral"], linewidth=1.7,
    label="Main anomaly score"
)
ax2.plot(
    window["date"], window["thr_main_adapt"],
    color=PALETTE["threshold"], linewidth=1.5, linestyle="--",
    label="Adaptive threshold"
)
ax2.scatter(
    [peak_date], [case_row["main_score"]],
    color=PALETTE["event"], s=70,
    edgecolor="#6B7280", linewidth=0.8, zorder=3
)
ax2.axvline(peak_date, color=PALETTE["event"], linestyle="--", linewidth=1.1)

ax2.set_ylabel("Score")
ax2.legend(loc="upper left")
soften_axes(ax2, "y")

ax2.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
for lab in ax2.get_xticklabels():
    lab.set_rotation(25)
    lab.set_ha("right")

# Panel 3: top-5 drivers
feature_colors = [PALETTE.get(g, PALETTE["other"]) for g in top_feat["feature_group"]]
ax3.barh(
    top_feat["feature_pretty"],
    top_feat["importance_norm"],
    color=feature_colors,
    edgecolor="white",
    linewidth=0.9,
    height=0.62,
)
ax3.invert_yaxis()
ax3.set_xlabel("Normalized importance")
ax3.text(
    0.0, 1.03, f"Top-5 numeric drivers | peak date = {peak_date.date()}",
    transform=ax3.transAxes,
    ha="left", va="bottom",
    fontsize=12, fontweight="semibold",
    color=PALETTE["text"]
)
soften_axes(ax3, "x")

fig.subplots_adjust(left=0.07, right=0.99, top=0.88, bottom=0.20, hspace=0.32, wspace=0.34)

save_figure(
    fig,
    "fig_08_detailed_grounded_case_study.png",
    caption=f"Figure 8. Detailed grounded anomaly case study for {ticker} on {peak_date.date()}",
    reserve_bottom=0.20,
)

print("Selected case:", ticker, peak_date.date())

[Saved figure] C:\Users\Abc\Downloads\finanomx\notebooks\report_outputs\figures\fig_08_detailed_grounded_case_study.png
Selected case: MSFT 2024-08-05


In [27]:
# 4.8 Table 6 - Comparison with prior works
metric_lookup = dict(zip(eval_explain["metric"], eval_explain["value"]))

finanomx_result = (
    f"Top-1 grounding confidence {metric_lookup.get('grounding_top1_mean_confidence', np.nan):.3f}; "
    f"full-text share {metric_lookup.get('grounding_top1_full_text_share', np.nan):.3f}; "
    f"strong counterfactual support share {metric_lookup.get('counterfactual_strong_support_share', np.nan):.3f}; "
    f"{len(grounded_all)} grounded peaks with packaged dashboard + chatbot artifacts."
)

comparison_df = pd.DataFrame([
    {
        "Study": "Li and Fan [1]",
        "Event / news grounding": "No article-level event grounding",
        "CF validation": "No",
        "Artifact availability": "Not the focus",
        "Representative result / contribution": "Anomaly attribution in financial networks, but not tied to article-level event evidence for one stock-day anomaly.",
    },
    {
        "Study": "Hu et al. (MindTS) [2]",
        "Event / news grounding": "Not finance-specific grounding",
        "CF validation": "No",
        "Artifact availability": "Not the focus",
        "Representative result / contribution": "Multimodal anomaly detection via time-text alignment, but not centered on finance-specific grounded cause identification.",
    },
    {
        "Study": "Çalık et al. [3]",
        "Event / news grounding": "No",
        "CF validation": "No",
        "Artifact availability": "Not the focus",
        "Representative result / contribution": "Prediction-oriented interpretability, but no grounded event retrieval and no rebuttal mechanism for anomaly causation.",
    },
    {
        "Study": "FinAnomX (this work)",
        "Event / news grounding": "Yes",
        "CF validation": "Yes",
        "Artifact availability": "Yes: dashboard + grounded chatbot + packaged artifacts",
        "Representative result / contribution": finanomx_result,
    },
])

save_table_csv(comparison_df, "table_06_comparison_with_prior_works.csv")
comparison_df

[Saved table] C:\Users\Abc\Downloads\finanomx\notebooks\report_outputs\tables\table_06_comparison_with_prior_works.csv


,Study,Event / news grounding,CF validation,Artifact availability,Representative result / contribution
0,Li and Fan [1],No article-level event grounding,No,Not the focus,"Anomaly attribution in financial networks, but..."
1,Hu et al. (MindTS) [2],Not finance-specific grounding,No,Not the focus,Multimodal anomaly detection via time-text ali...
2,Çalık et al. [3],No,No,Not the focus,"Prediction-oriented interpretability, but no g..."
3,FinAnomX (this work),Yes,Yes,Yes: dashboard + grounded chatbot + packaged a...,Top-1 grounding confidence 0.795; full-text sh...


In [ ]:
# 4.4 Figure 5b - Heatmap of XAI feature-group importance across grounded peaks (warm palette)
from matplotlib.colors import LinearSegmentedColormap

# --------------------------------------------------
# Chuẩn bị dữ liệu heatmap

# --------------------------------------------------
xai_top5 = xai_feature[xai_feature["rank"] <= 5].copy()

grounded_keys = grounded_analysis[["ticker", "date"]].drop_duplicates().copy()
xai_grounded = xai_top5.merge(grounded_keys, on=["ticker", "date"], how="inner")

heatmap_df = (
    xai_grounded.groupby(["ticker", "date", "feature_group"])["importance_norm"]
    .sum()
    .reset_index()
)

heatmap_pivot = (
    heatmap_df.pivot_table(
        index=["ticker", "date"],
        columns="feature_group",
        values="importance_norm",
        fill_value=0.0
    )
)

present_groups = [g for g in GROUP_ORDER if g in heatmap_pivot.columns]
heatmap_pivot = heatmap_pivot[present_groups].copy()

# Sắp xếp row theo ticker rồi theo date
heatmap_pivot = (
    heatmap_pivot.reset_index()
    .sort_values(["ticker", "date"])
    .set_index(["ticker", "date"])
)

row_labels = [
    f"{ticker} | {pd.to_datetime(date).date()}"
    for ticker, date in heatmap_pivot.index
]

# --------------------------------------------------
# Warm colormap: trắng -> vàng nhạt -> cam nhạt -> đỏ cam
# --------------------------------------------------
warm_cmap = LinearSegmentedColormap.from_list(
    "warm_red_orange",
    ["#FFFDF9", "#FCE9D8", "#F7C9A9", "#F39B6D", "#E76F51"]
)

# --------------------------------------------------
# Vẽ heatmap
# --------------------------------------------------
n_rows = len(heatmap_pivot)
fig_h = max(4.8, 0.34 * n_rows + 1.6)

fig, ax = plt.subplots(figsize=(8.4, fig_h))

im = ax.imshow(
    heatmap_pivot.values,
    aspect="auto",
    cmap=warm_cmap,
    vmin=0,
    vmax=max(0.35, float(np.nanmax(heatmap_pivot.values)))
)

# Tick labels
ax.set_xticks(np.arange(len(present_groups)))
ax.set_xticklabels(present_groups)

ax.set_yticks(np.arange(n_rows))
ax.set_yticklabels(row_labels)

ax.set_xlabel("Feature group")
ax.set_ylabel("Grounded episode peaks")

# Grid ô nhẹ
ax.set_xticks(np.arange(-0.5, len(present_groups), 1), minor=True)
ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
ax.grid(which="minor", color="#FFFFFF", linewidth=0.8)
ax.tick_params(which="minor", bottom=False, left=False)

# Separator giữa các ticker
ticker_sequence = [idx[0] for idx in heatmap_pivot.index]
break_positions = []
for i in range(1, len(ticker_sequence)):
    if ticker_sequence[i] != ticker_sequence[i - 1]:
        break_positions.append(i - 0.5)

for y in break_positions:
    ax.hlines(y, -0.5, len(present_groups) - 0.5, color="#D8C7BE", linewidth=1.1)

# Colorbar
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
cbar.set_label("Normalized importance")
cbar.outline.set_edgecolor("#D7DEE8")
cbar.outline.set_linewidth(0.8)

soften_axes(ax, grid_axis="none")

# Chừa khoảng trắng dưới cho caption
fig.subplots_adjust(left=0.27, right=0.92, top=0.97, bottom=0.14)

save_figure(
    fig,
    "fig_05b_xai_peak_feature_heatmap.png",
    caption="Figure 5b. Heatmap of XAI feature-group importance across grounded episode peaks",
    reserve_bottom=0.14,
)

heatmap_pivot.head()

[Saved figure] C:\Users\Abc\Downloads\finanomx\notebooks\report_outputs\figures\fig_05b_xai_peak_feature_heatmap.png


feature_group         price    volume  volatility  technical     macro
ticker date                                                           
AMZN   2023-10-27  0.387166  0.333402    0.195868   0.026950  0.000000
       2024-08-02  0.367288  0.079288    0.000000   0.047759  0.404048
MSFT   2024-02-01  0.017193  0.019112    0.026675   0.000000  0.899221
       2024-04-19  0.000000  0.376107    0.038084   0.466174  0.000000
       2024-04-25  0.122981  0.335499    0.000000   0.381612  0.000000